In [21]:
import logging
import pandas as pd
import numpy as np
import os
import glob
import zipfile
from io import BytesIO

In [ ]:
# Define the file path for your PDF file.
pdf_file_path = r'C:\Users\sbossier\Desktop\test\output.pdf'

In [7]:
os.environ['PDF_SERVICES_CLIENT_ID'] = '7979a1d8027346769c8eb40638c402ba'
os.environ['PDF_SERVICES_CLIENT_SECRET'] = 'p8e-w3yIxnNb3XoQNIkeCChG5AGp55VP9M7P'

from adobe.pdfservices.operation.auth.credentials import Credentials
from adobe.pdfservices.operation.exception.exceptions import ServiceApiException, ServiceUsageException, SdkException
from adobe.pdfservices.operation.pdfops.options.extractpdf.extract_pdf_options import ExtractPDFOptions
from adobe.pdfservices.operation.pdfops.options.extractpdf.extract_element_type import ExtractElementType
from adobe.pdfservices.operation.execution_context import ExecutionContext
from adobe.pdfservices.operation.io.file_ref import FileRef
from adobe.pdfservices.operation.pdfops.extract_pdf_operation import ExtractPDFOperation

logging.basicConfig(level=os.environ.get("LOGLEVEL", "INFO"))

try:
    # Initial setup, create credentials instance.
    credentials = Credentials.service_principal_credentials_builder(). \
        with_client_id(os.getenv('PDF_SERVICES_CLIENT_ID')). \
        with_client_secret(os.getenv('PDF_SERVICES_CLIENT_SECRET')). \
        build()

    # Create an ExecutionContext using credentials and create a new operation instance.
    execution_context = ExecutionContext.create(credentials)
    extract_pdf_operation = ExtractPDFOperation.create_new()

    # Set operation input from a source file.
    source = FileRef.create_from_local_file(pdf_file_path)
    extract_pdf_operation.set_input(source)
#.with_element_to_extract(ExtractElementType.TEXT) \
    # Build ExtractPDF options and set them into the operation
    extract_pdf_options: ExtractPDFOptions = ExtractPDFOptions.builder() \
        .with_element_to_extract(ExtractElementType.TABLES) \
        .build()
    extract_pdf_operation.set_options(extract_pdf_options)

    # Execute the operation.
    result: FileRef = extract_pdf_operation.execute(execution_context)

    # Define the path for the output file.
    output_file_path = os.path.dirname(pdf_file_path) + "/ExtractTextTableInfoFromPDF.zip"

    # Save the result to the specified location.
    result.save_as(output_file_path)
except (ServiceApiException, ServiceUsageException, SdkException):
    logging.exception("Exception encountered while executing operation")

INFO:adobe.pdfservices.operation.pdfops.extract_pdf_operation:All validations successfully done. Beginning ExtractPDF operation execution
INFO:root:Downloading file to C:\Users\sbossier\AppData\Local\Temp\sdk_result\39ed17a915c311eeb751e454e85f719c.zip
INFO:adobe.pdfservices.operation.pdfops.extract_pdf_operation:Extract Operation Successful - Transaction ID: 5cd6e711-d3e4-4127-88c6-23b65fa19437
INFO:adobe.pdfservices.operation.internal.io.file_ref_impl:Moving file at C:\Users\sbossier\AppData\Local\Temp\sdk_result\39ed17a915c311eeb751e454e85f719c.zip to target C:\Users\sbossier\Desktop\test/ExtractTextTableInfoFromPDF.zip


In [27]:
def extract_clean_and_concatenate_excel_from_zip(zip_file_path):
    dataframes = []
    
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        for file in zip_ref.namelist():
            if file.endswith('.xlsx') and 'tables' in file:  # replace 'subfolder' with actual subfolder name
                excel_data = zip_ref.read(file)

                # read the excel file, keep in memory using BytesIO
                excel_file = BytesIO(excel_data)
                df = pd.read_excel(excel_file, header=None)

                # clean the data
                cleaned_df = clean_data(df)

                # Append to list if the cleaned DataFrame is not empty
                if not cleaned_df.empty:
                    dataframes.append(cleaned_df)

    # Concatenate all the dataframes
    all_data = pd.concat(dataframes, ignore_index=True)

    # Set the column names
    all_data.columns = ['compound', 'RT', 'Exp. RT', 'area (a.u.)', 'conc']
    
    return all_data

In [26]:
# Function to clean data
def clean_data(df):
    # Replace '_x000D_' with a space in all cells
    df = df.astype(str).applymap(lambda x: x.replace('_x000D_', ''))
    
    # Remove leading and trailing whitespace
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    
    # Try to convert the data back to its original type
    df = df.apply(pd.to_numeric, errors='ignore')

    # Select only the rows starting with an empty cell or a specific string
    df = df[df.iloc[:, 0].isin(['nan', 'CH4', 'CO2', 'N2', 'CH3CH2OH', 'CH3OCH3', 'CH3OH', 'C2H6', 'C2H4', 'C2H2'])]

    # Convert 'nan' strings to actual NaN values
    df = df.replace('nan', np.nan)
    
    # Drop all-NaN rows
    df = df.dropna(axis=0, how='all')
    
    return df

In [29]:
# Using the function
zip_file_path = os.path.dirname(pdf_file_path) + "/ExtractTextTableInfoFromPDF.zip"
all_data = extract_clean_and_concatenate_excel_from_zip(zip_file_path)

# Now, all_data is a pandas DataFrame containing the cleaned and concatenated data from all Excel files.
# You can save it to a .csv file if you want:
all_data.to_csv(os.path.dirname(pdf_file_path) + '/combined.csv', index=False)